# 06A — Alternative Tabular Model Benchmark

This post-freeze experiment compares alternative tabular classifiers against the saved 05C LightGBM incident-held-out OOF reference.

Models:

- XGBoost
- CatBoost
- scikit-learn histogram gradient boosting
- elastic-net logistic regression trained with stochastic gradient descent
- Extra Trees
- Explainable Boosting Machine (EBM)

Scientific guardrails:

- The existing paper model remains frozen and is not replaced by this notebook.
- Every alternative uses the exact frozen 803-feature contract and the exact five incident-disjoint outer folds.
- Hyperparameters are prespecified; there is no tuning, calibration fitting, or threshold selection.
- OOF predictions are checkpointed by model and fold so interrupted Antigravity runs can resume.
- Comparisons use wildfire-incident bootstrap uncertainty.

The within-incident learning-to-rank model is intentionally left for a separate notebook because it answers a different prediction question and requires ranking-specific metrics.


## 0. Run configuration


In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "06A_V1_ALTERNATIVE_TABULAR_MODEL_BENCHMARK"

from pathlib import Path
import sys
import subprocess
import importlib.util

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))
MASTER_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_table_assembly_v1" / "fireline_hold_modeling_master_v1.parquet"
UNIT_DESIGN_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_design_v1" / "unit_outer_fold_and_training_weights_v1.parquet"
FEATURE_SETS_PATH = PROJECT_ROOT / "outputs" / "modeling_design_v1" / "FEATURE_SETS_V1.json"
LIGHTGBM_REFERENCE_PATH = PROJECT_ROOT / "data" / "processed" / "baseline_modeling_v1" / "oof_predictions" / "E_FULL_EXTENDED__row.parquet"
PRIMARY_PARAMS_PATH = PROJECT_ROOT / "models" / "fireline_hold_primary_v1" / "fireline_hold_primary_v1_params.json"
UPSTREAM_CHECKS_PATH = PROJECT_ROOT / "outputs" / "frozen_model_interpretation_v1" / "FINAL_FROZEN_MODEL_INTERPRETATION_CHECKS_V1.csv"

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "alternative_model_benchmark_v1"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "alternative_model_benchmark_v1"
PREDICTION_ROOT = PROCESSED_ROOT / "oof_prediction_shards"
MODEL_ROOT = PROJECT_ROOT / "models" / "alternative_model_benchmark_v1"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
for p in [OUTPUT_ROOT, PROCESSED_ROOT, PREDICTION_ROOT, MODEL_ROOT, FIGURE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

for p in [MASTER_PATH, UNIT_DESIGN_PATH, FEATURE_SETS_PATH, LIGHTGBM_REFERENCE_PATH, PRIMARY_PARAMS_PATH, UPSTREAM_CHECKS_PATH]:
    assert p.exists(), p

# "full" runs all six alternatives. "core" skips Extra Trees and EBM.
BENCHMARK_PROFILE = "full"
assert BENCHMARK_PROFILE in {"core", "full"}

CORE_MODELS = ["XGBOOST", "CATBOOST", "HIST_GRADIENT_BOOSTING", "ELASTIC_NET_LOGISTIC"]
EXTENDED_MODELS = ["EXTRA_TREES", "EBM"]
ENABLED_MODELS = CORE_MODELS + (EXTENDED_MODELS if BENCHMARK_PROFILE == "full" else [])

RESUME_COMPLETED_FOLDS = True
SAVE_FOLD_MODELS = False
INSTALL_MISSING_PACKAGES = True
BOOTSTRAP_REPLICATES = 5_000
RANDOM_SEED = 20260826
PREDICTION_EPS = 1e-6

EXPECTED_UNITS = 304_143
EXPECTED_INCIDENTS = 381
EXPECTED_FEATURES = 803
OUTER_FOLDS = 5

base_required = {"pandas": "pandas", "numpy": "numpy", "pyarrow": "pyarrow", "sklearn": "scikit-learn", "scipy": "scipy", "matplotlib": "matplotlib", "joblib": "joblib"}
optional_required = {"XGBOOST": ("xgboost", "xgboost"), "CATBOOST": ("catboost", "catboost"), "EBM": ("interpret", "interpret")}

for module, package in base_required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
for model_id in ENABLED_MODELS:
    if model_id in optional_required:
        module, package = optional_required[model_id]
        if importlib.util.find_spec(module) is None:
            if not INSTALL_MISSING_PACKAGES:
                raise ModuleNotFoundError(f"{module} is required for {model_id}")
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])

from datetime import datetime, timezone
import gc
import json
import platform
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn
from scipy.special import expit
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, log_loss
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

if "XGBOOST" in ENABLED_MODELS:
    import xgboost as xgb
if "CATBOOST" in ENABLED_MODELS:
    import catboost
    from catboost import CatBoostClassifier
if "EBM" in ENABLED_MODELS:
    import interpret
    from interpret.glassbox import ExplainableBoostingClassifier

pd.set_option("display.max_columns", 400)
pd.set_option("display.max_rows", 400)
pd.set_option("display.width", 420)
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 240, "axes.spines.top": False, "axes.spines.right": False})

print("=" * 100)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("Profile:", BENCHMARK_PROFILE)
print("Enabled alternatives:", ENABLED_MODELS)
print("Resume completed folds:", RESUME_COMPLETED_FOLDS)
print("Save fold models:", SAVE_FOLD_MODELS)
print("=" * 100)


## 1. Fixed model specifications and software versions


In [ ]:
MODEL_SPECS = {
    "XGBOOST": {
        "objective": "binary:logistic", "n_estimators": 500, "learning_rate": 0.05,
        "max_depth": 8, "min_child_weight": 20.0, "subsample": 0.85,
        "colsample_bytree": 0.80, "reg_alpha": 0.0, "reg_lambda": 1.0,
        "tree_method": "hist", "max_bin": 127, "eval_metric": "logloss", "n_jobs": -1,
    },
    "CATBOOST": {
        "loss_function": "Logloss", "iterations": 500, "learning_rate": 0.05,
        "depth": 8, "l2_leaf_reg": 5.0, "random_strength": 1.0,
        "rsm": 0.80, "border_count": 127, "thread_count": -1,
        "verbose": False, "allow_writing_files": False,
    },
    "HIST_GRADIENT_BOOSTING": {
        "loss": "log_loss", "learning_rate": 0.05, "max_iter": 300,
        "max_leaf_nodes": 31, "max_depth": None, "min_samples_leaf": 150,
        "l2_regularization": 1.0, "max_bins": 127, "early_stopping": False,
    },
    "ELASTIC_NET_LOGISTIC": {
        "loss": "log_loss", "penalty": "elasticnet", "alpha": 1e-5,
        "l1_ratio": 0.05, "max_iter": 2_000, "tol": 1e-4,
        "average": True, "learning_rate": "optimal", "n_jobs": -1,
    },
    "EXTRA_TREES": {
        "n_estimators": 300, "criterion": "log_loss", "max_depth": 24,
        "min_samples_leaf": 75, "max_features": "sqrt", "bootstrap": False,
        "n_jobs": -1,
    },
    "EBM": {
        "max_bins": 128, "interactions": 10, "outer_bags": 8, "inner_bags": 0,
        "learning_rate": 0.03, "max_rounds": 2_000, "min_samples_leaf": 50,
        "max_leaves": 3, "n_jobs": -1,
    },
}

versions = [
    {"package": "python", "version": sys.version.split()[0]},
    {"package": "numpy", "version": np.__version__},
    {"package": "pandas", "version": pd.__version__},
    {"package": "scikit-learn", "version": sklearn.__version__},
]
if "XGBOOST" in ENABLED_MODELS:
    versions.append({"package": "xgboost", "version": xgb.__version__})
if "CATBOOST" in ENABLED_MODELS:
    versions.append({"package": "catboost", "version": catboost.__version__})
if "EBM" in ENABLED_MODELS:
    versions.append({"package": "interpret", "version": interpret.__version__})
software_versions = pd.DataFrame(versions)
software_versions.to_csv(OUTPUT_ROOT / "SOFTWARE_VERSIONS_V1.csv", index=False)
(OUTPUT_ROOT / "ALTERNATIVE_MODEL_FIXED_SPECS_V1.json").write_text(json.dumps(MODEL_SPECS, indent=2), encoding="utf-8")
display(software_versions)
display(pd.DataFrame([{"model_id": k, **v} for k, v in MODEL_SPECS.items() if k in ENABLED_MODELS]))


## 2. Load the exact frozen design and 803-feature matrix


In [ ]:
with FEATURE_SETS_PATH.open("r", encoding="utf-8") as f:
    feature_sets = json.load(f)
with PRIMARY_PARAMS_PATH.open("r", encoding="utf-8") as f:
    primary_params = json.load(f)

features = feature_sets["E_FULL_EXTENDED"]
assert len(features) == EXPECTED_FEATURES and len(features) == len(set(features))

base_columns = ["unit_uid", "_irwin_norm", "IncidentName", "Year", "hold_label", "outer_fold", "weight_row_raw"]
design = pd.read_parquet(UNIT_DESIGN_PATH, columns=base_columns)
reference = pd.read_parquet(LIGHTGBM_REFERENCE_PATH, columns=["unit_uid", "pred_prob"])
feature_table = pd.read_parquet(MASTER_PATH, columns=["unit_uid", *features])

assert len(design) == EXPECTED_UNITS and design["unit_uid"].is_unique
assert design["_irwin_norm"].nunique() == EXPECTED_INCIDENTS
assert set(design["outer_fold"].unique()) == set(range(OUTER_FOLDS))
assert (design.groupby("_irwin_norm")["outer_fold"].nunique() == 1).all()
assert len(reference) == EXPECTED_UNITS and reference["unit_uid"].is_unique
assert len(feature_table) == EXPECTED_UNITS and feature_table["unit_uid"].is_unique

frame = design.merge(reference.rename(columns={"pred_prob": "lightgbm_reference_oof"}), on="unit_uid", how="left", validate="one_to_one")
feature_table = design[["unit_uid"]].merge(feature_table, on="unit_uid", how="left", validate="one_to_one")

X = pd.DataFrame(index=feature_table.index)
for feature in features:
    X[feature] = pd.to_numeric(feature_table[feature], errors="coerce").astype(np.float32)
del feature_table
gc.collect()

y = frame["hold_label"].to_numpy(np.int8)
folds = frame["outer_fold"].to_numpy(int)
row_weights = frame["weight_row_raw"].to_numpy(float)
row_weights = row_weights * (len(row_weights) / row_weights.sum())
reference_oof = frame["lightgbm_reference_oof"].to_numpy(float)

assert X.shape == (EXPECTED_UNITS, EXPECTED_FEATURES)
assert np.isfinite(y).all() and set(np.unique(y)) == {0, 1}
assert np.isfinite(row_weights).all() and (row_weights > 0).all() and np.isclose(row_weights.mean(), 1.0)
assert np.isfinite(reference_oof).all() and ((reference_oof >= 0) & (reference_oof <= 1)).all()

print("Feature matrix shape:", X.shape)
print("Feature matrix GiB:", round(X.memory_usage(index=True, deep=True).sum() / 1024**3, 3))
print("Units/incidents:", len(frame), frame["_irwin_norm"].nunique())


## 3. Preprocessing, estimator adapters, and checkpoint validation


In [ ]:
def clip_prob(values):
    return np.clip(np.asarray(values, dtype=float), PREDICTION_EPS, 1.0 - PREDICTION_EPS)


def median_impute_float32(train_frame, test_frame):
    medians = train_frame.median(axis=0, skipna=True).fillna(0.0).to_numpy(np.float32)
    train_array = train_frame.to_numpy(dtype=np.float32, copy=True)
    test_array = test_frame.to_numpy(dtype=np.float32, copy=True)
    train_missing = ~np.isfinite(train_array)
    test_missing = ~np.isfinite(test_array)
    if train_missing.any():
        train_array[train_missing] = medians[np.where(train_missing)[1]]
    if test_missing.any():
        test_array[test_missing] = medians[np.where(test_missing)[1]]
    return train_array, test_array, medians


def build_native_estimator(model_id, seed):
    params = dict(MODEL_SPECS[model_id])
    if model_id == "XGBOOST":
        params["random_state"] = seed
        return xgb.XGBClassifier(**params)
    if model_id == "CATBOOST":
        params["random_seed"] = seed
        return CatBoostClassifier(**params)
    if model_id == "HIST_GRADIENT_BOOSTING":
        params["random_state"] = seed
        return HistGradientBoostingClassifier(**params)
    if model_id == "EXTRA_TREES":
        params["random_state"] = seed
        return ExtraTreesClassifier(**params)
    if model_id == "EBM":
        params["random_state"] = seed
        return ExplainableBoostingClassifier(**params)
    raise KeyError(model_id)


def fit_predict_one_fold(model_id, train_idx, test_idx, seed):
    x_train = X.iloc[train_idx]
    x_test = X.iloc[test_idx]
    y_train = y[train_idx]
    weights_train = row_weights[train_idx]
    preprocessing_seconds = 0.0

    if model_id == "ELASTIC_NET_LOGISTIC":
        prep_start = time.time()
        train_array, test_array, medians = median_impute_float32(x_train, x_test)
        scaler = StandardScaler(copy=False)
        train_array = scaler.fit_transform(train_array).astype(np.float32, copy=False)
        test_array = scaler.transform(test_array).astype(np.float32, copy=False)
        preprocessing_seconds = time.time() - prep_start
        params = dict(MODEL_SPECS[model_id])
        params["random_state"] = seed
        model = SGDClassifier(**params)
        fit_start = time.time()
        model.fit(train_array, y_train, sample_weight=weights_train)
        fit_seconds = time.time() - fit_start
        predict_start = time.time()
        pred = model.predict_proba(test_array)[:, 1]
        predict_seconds = time.time() - predict_start
        del train_array, test_array, medians, scaler
    elif model_id == "EXTRA_TREES":
        prep_start = time.time()
        train_array, test_array, medians = median_impute_float32(x_train, x_test)
        preprocessing_seconds = time.time() - prep_start
        model = build_native_estimator(model_id, seed)
        fit_start = time.time()
        model.fit(train_array, y_train, sample_weight=weights_train)
        fit_seconds = time.time() - fit_start
        predict_start = time.time()
        pred = model.predict_proba(test_array)[:, 1]
        predict_seconds = time.time() - predict_start
        del train_array, test_array, medians
    else:
        model = build_native_estimator(model_id, seed)
        fit_start = time.time()
        model.fit(x_train, y_train, sample_weight=weights_train)
        fit_seconds = time.time() - fit_start
        predict_start = time.time()
        pred = model.predict_proba(x_test)[:, 1]
        predict_seconds = time.time() - predict_start

    pred = clip_prob(pred)
    assert len(pred) == len(test_idx) and np.isfinite(pred).all()
    return model, pred, preprocessing_seconds, fit_seconds, predict_seconds


def checkpoint_path(model_id, outer_fold):
    return PREDICTION_ROOT / model_id / f"{model_id}__outer_fold_{outer_fold}.parquet"


def valid_checkpoint(path, model_id, outer_fold, expected_uids):
    if not path.exists():
        return False
    try:
        saved = pd.read_parquet(path)
    except Exception:
        return False
    required_columns = {"unit_uid", "_irwin_norm", "hold_label", "outer_fold", "pred_prob", "model_id"}
    return (
        required_columns.issubset(saved.columns)
        and len(saved) == len(expected_uids)
        and saved["unit_uid"].is_unique
        and set(saved["unit_uid"]) == set(expected_uids)
        and saved["model_id"].eq(model_id).all()
        and saved["outer_fold"].eq(outer_fold).all()
        and np.isfinite(saved["pred_prob"]).all()
    )


## 4. Train or resume every alternative model across the five incident folds


In [ ]:
runtime_rows = []
completed_prediction_paths = []

for model_position, model_id in enumerate(ENABLED_MODELS):
    (PREDICTION_ROOT / model_id).mkdir(parents=True, exist_ok=True)
    (MODEL_ROOT / model_id).mkdir(parents=True, exist_ok=True)
    print("=" * 100)
    print("MODEL:", model_id)
    print("=" * 100)

    for outer_fold in range(OUTER_FOLDS):
        train_idx = np.flatnonzero(folds != outer_fold)
        test_idx = np.flatnonzero(folds == outer_fold)
        train_incidents = set(frame.iloc[train_idx]["_irwin_norm"])
        test_incidents = set(frame.iloc[test_idx]["_irwin_norm"])
        assert train_incidents.isdisjoint(test_incidents)

        path = checkpoint_path(model_id, outer_fold)
        expected_uids = frame.iloc[test_idx]["unit_uid"].tolist()
        if RESUME_COMPLETED_FOLDS and valid_checkpoint(path, model_id, outer_fold, expected_uids):
            saved = pd.read_parquet(path)
            runtime_rows.append({
                "model_id": model_id, "outer_fold": outer_fold, "status": "resumed",
                "train_units": len(train_idx), "train_incidents": len(train_incidents),
                "test_units": len(test_idx), "test_incidents": len(test_incidents),
                "preprocessing_seconds": np.nan, "fit_seconds": np.nan, "predict_seconds": np.nan,
                "prediction_path": str(path),
            })
            completed_prediction_paths.append(path)
            print(f"Fold {outer_fold}: resumed {len(saved):,} predictions")
            continue

        seed = RANDOM_SEED + model_position * 100 + outer_fold
        print(f"Fold {outer_fold}: training on {len(train_idx):,} units / {len(train_incidents)} incidents")
        try:
            model, pred, prep_seconds, fit_seconds, predict_seconds = fit_predict_one_fold(model_id, train_idx, test_idx, seed)
        except Exception as exc:
            error_row = pd.DataFrame([{
                "generated_utc": datetime.now(timezone.utc).isoformat(),
                "model_id": model_id, "outer_fold": outer_fold,
                "exception_type": type(exc).__name__, "exception_message": str(exc),
            }])
            error_path = OUTPUT_ROOT / "BENCHMARK_ERRORS_V1.csv"
            if error_path.exists():
                previous = pd.read_csv(error_path)
                error_row = pd.concat([previous, error_row], ignore_index=True)
            error_row.to_csv(error_path, index=False)
            raise

        shard = frame.iloc[test_idx][["unit_uid", "_irwin_norm", "IncidentName", "Year", "hold_label", "outer_fold"]].copy()
        shard["model_id"] = model_id
        shard["pred_prob"] = pred
        shard.to_parquet(path, index=False)
        assert valid_checkpoint(path, model_id, outer_fold, expected_uids)
        completed_prediction_paths.append(path)

        if SAVE_FOLD_MODELS:
            model_path = MODEL_ROOT / model_id / f"{model_id}__outer_fold_{outer_fold}.joblib"
            joblib.dump(model, model_path, compress=3)

        runtime_rows.append({
            "model_id": model_id, "outer_fold": outer_fold, "status": "trained",
            "train_units": len(train_idx), "train_incidents": len(train_incidents),
            "test_units": len(test_idx), "test_incidents": len(test_incidents),
            "preprocessing_seconds": prep_seconds, "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds, "prediction_path": str(path),
        })
        pd.DataFrame(runtime_rows).to_csv(OUTPUT_ROOT / "ALTERNATIVE_MODEL_RUNTIME_V1.csv", index=False)
        print(f"Fold {outer_fold}: prep={prep_seconds/60:.2f}m fit={fit_seconds/60:.2f}m predict={predict_seconds/60:.2f}m")
        del model, pred, shard
        gc.collect()

runtime = pd.DataFrame(runtime_rows)
runtime.to_csv(OUTPUT_ROOT / "ALTERNATIVE_MODEL_RUNTIME_V1.csv", index=False)
display(runtime)


## 5. Assemble complete OOF predictions and calculate identical metrics


In [ ]:
def prediction_summary(prediction_frame):
    y_true = prediction_frame["hold_label"].to_numpy(int)
    pred = clip_prob(prediction_frame["pred_prob"].to_numpy(float))
    incident_rows = []
    for incident, g in prediction_frame.groupby("_irwin_norm", sort=False):
        yi = g["hold_label"].to_numpy(int)
        pi = clip_prob(g["pred_prob"].to_numpy(float))
        mixed = np.unique(yi).size == 2
        incident_rows.append({
            "_irwin_norm": incident,
            "units": len(g),
            "held_rate": float(np.mean(yi)),
            "brier": float(np.mean((pi - yi) ** 2)),
            "log_loss": float(log_loss(yi, pi, labels=[0, 1])),
            "mixed_class": mixed,
            "pr_auc": float(average_precision_score(yi, pi)) if mixed else np.nan,
            "roc_auc": float(roc_auc_score(yi, pi)) if mixed else np.nan,
        })
    incident = pd.DataFrame(incident_rows)
    mixed = incident["mixed_class"]
    summary = {
        "units": len(prediction_frame),
        "incidents": prediction_frame["_irwin_norm"].nunique(),
        "held_rate": float(np.mean(y_true)),
        "mean_pred": float(np.mean(pred)),
        "pooled_pr_auc": float(average_precision_score(y_true, pred)),
        "pooled_roc_auc": float(roc_auc_score(y_true, pred)),
        "pooled_brier": float(brier_score_loss(y_true, pred)),
        "pooled_log_loss": float(log_loss(y_true, pred, labels=[0, 1])),
        "incident_macro_brier": float(incident["brier"].mean()),
        "incident_macro_log_loss": float(incident["log_loss"].mean()),
        "mixed_incidents": int(mixed.sum()),
        "mixed_incident_macro_pr_auc": float(incident.loc[mixed, "pr_auc"].mean()),
        "mixed_incident_macro_roc_auc": float(incident.loc[mixed, "roc_auc"].mean()),
    }
    return summary, incident


prediction_frames = []
reference_frame = frame[["unit_uid", "_irwin_norm", "IncidentName", "Year", "hold_label", "outer_fold"]].copy()
reference_frame["model_id"] = "LIGHTGBM_05C_REFERENCE"
reference_frame["pred_prob"] = reference_oof
prediction_frames.append(reference_frame)

for model_id in ENABLED_MODELS:
    model_parts = [pd.read_parquet(checkpoint_path(model_id, fold)) for fold in range(OUTER_FOLDS)]
    model_oof = pd.concat(model_parts, ignore_index=True)
    assert len(model_oof) == EXPECTED_UNITS and model_oof["unit_uid"].is_unique
    assert set(model_oof["unit_uid"]) == set(frame["unit_uid"])
    prediction_frames.append(model_oof)

all_oof = pd.concat(prediction_frames, ignore_index=True)
all_oof.to_parquet(PROCESSED_ROOT / "ALTERNATIVE_MODEL_ALL_OOF_PREDICTIONS_V1.parquet", index=False)

summary_rows = []
incident_metric_frames = []
fold_rows = []
for model_id, g in all_oof.groupby("model_id", sort=False):
    summary, incident = prediction_summary(g)
    summary_rows.append({"model_id": model_id, **summary})
    incident["model_id"] = model_id
    incident_metric_frames.append(incident)
    for outer_fold, q in g.groupby("outer_fold", sort=True):
        fold_summary, _ = prediction_summary(q)
        fold_rows.append({"model_id": model_id, "outer_fold": int(outer_fold), **fold_summary})

model_metrics = pd.DataFrame(summary_rows).sort_values("incident_macro_brier")
incident_metrics = pd.concat(incident_metric_frames, ignore_index=True)
fold_metrics = pd.DataFrame(fold_rows)
model_metrics.to_csv(OUTPUT_ROOT / "ALTERNATIVE_MODEL_OOF_METRICS_V1.csv", index=False)
incident_metrics.to_parquet(PROCESSED_ROOT / "ALTERNATIVE_MODEL_INCIDENT_METRICS_V1.parquet", index=False)
fold_metrics.to_csv(OUTPUT_ROOT / "ALTERNATIVE_MODEL_OUTER_FOLD_METRICS_V1.csv", index=False)
display(model_metrics)


## 6. Paired wildfire-incident bootstrap versus the frozen LightGBM reference


In [ ]:
REFERENCE_MODEL_ID = "LIGHTGBM_05C_REFERENCE"
BOOTSTRAP_METRICS = {
    "brier": "lower",
    "log_loss": "lower",
    "pr_auc": "higher",
    "roc_auc": "higher",
}

reference_incident = incident_metrics[incident_metrics["model_id"] == REFERENCE_MODEL_ID].set_index("_irwin_norm")
bootstrap_rows = []
rng = np.random.default_rng(RANDOM_SEED + 9_000)

for model_id in ENABLED_MODELS:
    alternative_incident = incident_metrics[incident_metrics["model_id"] == model_id].set_index("_irwin_norm")
    common = reference_incident.index.intersection(alternative_incident.index)
    assert len(common) == EXPECTED_INCIDENTS
    for metric, direction in BOOTSTRAP_METRICS.items():
        ref_values = reference_incident.loc[common, metric].to_numpy(float)
        alt_values = alternative_incident.loc[common, metric].to_numpy(float)
        valid = np.isfinite(ref_values) & np.isfinite(alt_values)
        differences = alt_values[valid] - ref_values[valid]
        sample_indices = rng.integers(0, len(differences), size=(BOOTSTRAP_REPLICATES, len(differences)))
        bootstrap_delta = differences[sample_indices].mean(axis=1)
        probability_better = float(np.mean(bootstrap_delta < 0)) if direction == "lower" else float(np.mean(bootstrap_delta > 0))
        bootstrap_rows.append({
            "alternative_model": model_id,
            "reference_model": REFERENCE_MODEL_ID,
            "metric": metric,
            "direction_better": direction,
            "incidents_evaluated": int(valid.sum()),
            "bootstrap_replicates": BOOTSTRAP_REPLICATES,
            "delta_alt_minus_reference_mean": float(differences.mean()),
            "ci_low": float(np.quantile(bootstrap_delta, 0.025)),
            "ci_high": float(np.quantile(bootstrap_delta, 0.975)),
            "probability_alternative_better": probability_better,
        })

paired_bootstrap = pd.DataFrame(bootstrap_rows)
paired_bootstrap.to_csv(OUTPUT_ROOT / "PAIRED_INCIDENT_BOOTSTRAP_VS_LIGHTGBM_V1.csv", index=False)
display(paired_bootstrap)


## 7. Comparison figures


In [ ]:
MODEL_COLORS = {
    "LIGHTGBM_05C_REFERENCE": "#4c72b0", "XGBOOST": "#dd8452", "CATBOOST": "#55a868",
    "HIST_GRADIENT_BOOSTING": "#c44e52", "ELASTIC_NET_LOGISTIC": "#8172b3",
    "EXTRA_TREES": "#937860", "EBM": "#64b5cd",
}

def save_figure(fig, stem):
    fig.savefig(FIGURE_ROOT / f"{stem}.png", bbox_inches="tight")
    fig.savefig(FIGURE_ROOT / f"{stem}.pdf", bbox_inches="tight")


ordered = model_metrics.sort_values("mixed_incident_macro_roc_auc")
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), constrained_layout=True)
axes[0].barh(ordered["model_id"], ordered["mixed_incident_macro_roc_auc"], color=[MODEL_COLORS[m] for m in ordered["model_id"]])
axes[0].axvline(0.5, color="black", linestyle="--", linewidth=1)
axes[0].set_xlabel("Mixed-incident macro ROC-AUC")
axes[0].set_title("Within-incident discrimination")
ordered_loss = model_metrics.sort_values("incident_macro_brier", ascending=False)
axes[1].barh(ordered_loss["model_id"], ordered_loss["incident_macro_brier"], color=[MODEL_COLORS[m] for m in ordered_loss["model_id"]])
axes[1].set_xlabel("Incident-macro Brier score (lower is better)")
axes[1].set_title("Incident-balanced probability loss")
for ax in axes:
    ax.grid(axis="x", alpha=0.2)
fig.suptitle("Alternative tabular models under identical incident-held-out folds", fontsize=15)
save_figure(fig, "FIG_06A_01_MODEL_PERFORMANCE_COMPARISON_V1")
plt.show()

plot_delta = paired_bootstrap[paired_bootstrap["metric"].isin(["brier", "roc_auc"])].copy()
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), constrained_layout=True)
for ax, metric, title in zip(axes, ["brier", "roc_auc"], ["Incident-macro Brier delta", "Mixed-incident macro ROC-AUC delta"]):
    q = plot_delta[plot_delta["metric"] == metric].sort_values("delta_alt_minus_reference_mean")
    y_pos = np.arange(len(q))
    x = q["delta_alt_minus_reference_mean"].to_numpy(float)
    xerr = np.vstack([x - q["ci_low"].to_numpy(float), q["ci_high"].to_numpy(float) - x])
    ax.errorbar(x, y_pos, xerr=xerr, fmt="o", color="#4c4c4c", capsize=3)
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_yticks(y_pos, q["alternative_model"])
    ax.set_title(title)
    ax.set_xlabel("Alternative minus LightGBM (95% incident-bootstrap CI)")
    ax.grid(axis="x", alpha=0.2)
save_figure(fig, "FIG_06A_02_PAIRED_DELTAS_VS_LIGHTGBM_V1")
plt.show()

runtime_trained = runtime[runtime["status"] == "trained"].groupby("model_id", as_index=False).agg(
    fit_minutes=("fit_seconds", lambda x: float(np.nansum(x) / 60)),
    preprocessing_minutes=("preprocessing_seconds", lambda x: float(np.nansum(x) / 60)),
)
runtime_plot = runtime_trained.merge(model_metrics[["model_id", "incident_macro_brier", "mixed_incident_macro_roc_auc"]], on="model_id", how="left")
fig, ax = plt.subplots(figsize=(8, 6.5), constrained_layout=True)
for _, row in runtime_plot.iterrows():
    ax.scatter(row["fit_minutes"], row["incident_macro_brier"], s=80, color=MODEL_COLORS[row["model_id"]])
    ax.annotate(row["model_id"], (row["fit_minutes"], row["incident_macro_brier"]), xytext=(5, 4), textcoords="offset points", fontsize=8)
ax.set_xlabel("Total five-fold fit time (minutes; current run only)")
ax.set_ylabel("Incident-macro Brier score")
ax.set_title("Compute–performance tradeoff")
ax.grid(alpha=0.2)
save_figure(fig, "FIG_06A_03_RUNTIME_VS_PERFORMANCE_V1")
plt.show()

fig, ax = plt.subplots(figsize=(10, 6.5), constrained_layout=True)
for model_id, g in fold_metrics.groupby("model_id"):
    q = g.sort_values("outer_fold")
    ax.plot(q["outer_fold"], q["pooled_roc_auc"], marker="o", linewidth=1.8, label=model_id, color=MODEL_COLORS[model_id])
ax.set_xticks(range(OUTER_FOLDS))
ax.set_xlabel("Frozen outer fold")
ax.set_ylabel("Fold pooled ROC-AUC")
ax.set_title("Outer-fold performance variability")
ax.legend(frameon=False, fontsize=8, ncol=2)
ax.grid(alpha=0.2)
save_figure(fig, "FIG_06A_04_OUTER_FOLD_VARIABILITY_V1")
plt.show()


## 8. Integrity checks and benchmark report


In [ ]:
expected_models = {"LIGHTGBM_05C_REFERENCE", *ENABLED_MODELS}
figure_pngs = sorted(FIGURE_ROOT.glob("FIG_06A_*.png"))
figure_pdfs = sorted(FIGURE_ROOT.glob("FIG_06A_*.pdf"))
checks = pd.DataFrame([
    {"check": "upstream_frozen_interpretation_checks_pass", "pass": bool(pd.read_csv(UPSTREAM_CHECKS_PATH)["pass"].astype(bool).all())},
    {"check": "exact_304143_units", "pass": len(frame) == EXPECTED_UNITS},
    {"check": "exact_381_incidents", "pass": frame["_irwin_norm"].nunique() == EXPECTED_INCIDENTS},
    {"check": "exact_803_frozen_features", "pass": len(features) == EXPECTED_FEATURES},
    {"check": "exact_five_incident_disjoint_folds", "pass": set(frame["outer_fold"].unique()) == set(range(OUTER_FOLDS)) and bool((frame.groupby("_irwin_norm")["outer_fold"].nunique() == 1).all())},
    {"check": "all_enabled_model_fold_checkpoints_complete", "pass": len(completed_prediction_paths) == len(ENABLED_MODELS) * OUTER_FOLDS},
    {"check": "all_models_have_exact_complete_oof_units", "pass": set(all_oof["model_id"]) == expected_models and bool(all_oof.groupby("model_id")["unit_uid"].nunique().eq(EXPECTED_UNITS).all())},
    {"check": "all_oof_predictions_finite_and_bounded", "pass": bool(np.isfinite(all_oof["pred_prob"]).all() and all_oof["pred_prob"].between(0, 1).all())},
    {"check": "all_models_use_same_binary_targets", "pass": bool(all_oof.groupby("model_id")["hold_label"].mean().nunique() == 1)},
    {"check": "all_model_metrics_finite", "pass": bool(np.isfinite(model_metrics.select_dtypes(include=[np.number])).all().all())},
    {"check": "paired_incident_bootstrap_5000_complete", "pass": bool(paired_bootstrap["bootstrap_replicates"].eq(BOOTSTRAP_REPLICATES).all())},
    {"check": "exact_four_png_and_four_pdf_figures", "pass": len(figure_pngs) == 4 and len(figure_pdfs) == 4},
    {"check": "prespecified_fixed_model_specs_no_tuning", "pass": True},
    {"check": "same_ordinary_row_weighting_as_05c", "pass": bool(np.isclose(row_weights.mean(), 1.0))},
    {"check": "no_posthoc_calibration", "pass": True},
    {"check": "no_probability_threshold_selection", "pass": True},
    {"check": "paper_primary_remains_frozen_lightgbm", "pass": True},
])
PASS = bool(checks["pass"].all())
checks.to_csv(OUTPUT_ROOT / "FINAL_ALTERNATIVE_MODEL_BENCHMARK_CHECKS_V1.csv", index=False)
display(checks)


def md_table(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)


report_lines = [
    "# 06A Alternative Tabular Model Benchmark",
    "",
    f"Build: `{NOTEBOOK_BUILD_ID}`",
    "",
    f"Generated UTC: {datetime.now(timezone.utc).isoformat()}",
    "",
    "## Scientific status",
    "",
    "Post-freeze model-family robustness experiment. The paper's frozen 05C LightGBM primary model was not changed.",
    "",
    "All alternatives used the exact 803-feature contract, ordinary row weighting, and five incident-disjoint outer folds. Hyperparameters were prespecified; no tuning, recalibration, or threshold selection occurred.",
    "",
    "## Model specifications",
    "",
    "```json",
    json.dumps({k: MODEL_SPECS[k] for k in ENABLED_MODELS}, indent=2),
    "```",
    "",
    "## OOF model comparison",
    "",
    md_table(model_metrics),
    "",
    "## Paired incident bootstrap versus frozen LightGBM",
    "",
    md_table(paired_bootstrap),
    "",
    "## Runtime",
    "",
    md_table(runtime),
    "",
    "## Final checks",
    "",
    md_table(checks),
    "",
    "## Decision",
    "",
    "`PASS / ALTERNATIVE MODEL BENCHMARK COMPLETE; PAPER PRIMARY REMAINS FROZEN`" if PASS else "`REVIEW REQUIRED`",
]
REPORT_PATH = OUTPUT_ROOT / "ALTERNATIVE_MODEL_BENCHMARK_REPORT_V1.md"
REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")

print()
print("06A ALTERNATIVE MODEL BENCHMARK:", "PASS" if PASS else "REVIEW REQUIRED")
print("Saved:", REPORT_PATH)
print("DONE:", NOTEBOOK_BUILD_ID)


## 9. Output inventory


In [ ]:
print("=" * 100)
print("06A OUTPUTS")
print("=" * 100)
for root in [OUTPUT_ROOT, PROCESSED_ROOT]:
    print()
    print(root)
    for p in sorted(root.rglob("*")):
        if p.is_file():
            print(f"  {p.relative_to(root)!s:<95} {p.stat().st_size/(1024**2):>10.3f} MB")
